# Goal 3 — Consultas, atualizações e transações em MySQL

## Consultas em MySQL
Consultas com diferentes níveis de complexidade.

In [8]:
# %pip install mysql-connector-python pandas

from concurrent.futures import ThreadPoolExecutor
from threading import Barrier
from time import perf_counter
import pandas as pd

In [9]:
import mysql.connector
import pandas as pd
from getpass import getpass

connection = mysql.connector.connect(host="localhost", port=3306, user="root", password=getpass("Password MySQL: "), database="bda_movielens")

def consultar(sql):
    with connection.cursor() as cursor:
        cursor.execute(sql)
        return pd.DataFrame(cursor.fetchall(), columns=cursor.column_names)

## Q1 — Consulta simples
Filtragem e projeção. Consultar os filmes lançados após 2010.

In [10]:
q1 = """
SELECT movieId, title, year
FROM movies
WHERE year > 2010
ORDER BY movieId ASC;
"""

resultado_q1 = consultar(q1) 

print("Número de resultados:", len(q1))
display(resultado_q1)

Número de resultados: 81


,movieId,title,year
0,82152,Beastly,2011
1,83349,"Green Hornet, The",2011
2,83480,Season of the Witch,2011
3,83613,Cowboys & Aliens,2011
4,83910,"Dilemma, The",2011
...,...,...,...
1679,193579,Jon Stewart Has Left the Building,2015
1680,193581,Black Butler: Book of the Atlantic,2017
1681,193583,No Game No Life: Zero,2017
1682,193585,Flint,2017


## Q2 — Consulta básica
Filtragem, projeção e ordenação. Consultar os 20 filmes mais recentes.

In [11]:
q2 = """
SELECT movieId, title, year
FROM movies
WHERE year IS NOT NULL
ORDER BY year DESC, movieId ASC
LIMIT 20;
"""
resultado_q2 = consultar(q2)

print("Número de resultados:", len(q2))
display(resultado_q2)

Número de resultados: 106


,movieId,title,year
0,122912,Avengers: Infinity War - Part I,2018
1,182715,Annihilation,2018
2,183011,The Commuter,2018
3,183295,Insidious: The Last Key,2018
4,183611,Game Night,2018
5,183635,Maze Runner: The Death Cure,2018
6,183897,Isle of Dogs,2018
7,183911,The Clapper,2018
8,183959,Tom Segura: Disgraceful,2018
9,184015,When We First Met,2018


## Q3 — Consulta intermédia
Junção, agregação e agrupamento. Calcular a classificação média e o número de avaliações de cada filme avaliado.

In [12]:
q3 ="""
SELECT m.movieId, m.title,
    ROUND(AVG(r.rating), 4) AS averageRating,
    COUNT(*) AS totalRatings
FROM movies AS m
INNER JOIN ratings AS r ON m.movieId = r.movieId
GROUP BY m.movieId, m.title
ORDER BY m.movieId ASC;
"""
resultado_q3 = consultar(q3)

print("Número de resultados:", len(q3))
display(resultado_q3)

Número de resultados: 221


,movieId,title,averageRating,totalRatings
0,1,Toy Story,3.9206,214
1,2,Jumanji,3.4312,109
2,3,Grumpier Old Men,3.2596,52
3,4,Waiting to Exhale,2.3571,7
4,5,Father of the Bride Part II,3.0714,49
...,...,...,...,...
9688,193581,Black Butler: Book of the Atlantic,4.0000,1
9689,193583,No Game No Life: Zero,3.5000,1
9690,193585,Flint,3.5000,1
9691,193587,Bungo Stray Dogs: Dead Apple,3.5000,1


## Q4 — Consulta Moderada
Junção, agregação, agrupamento, coordenadas. Calcular e ordenar a classificação média de classificações num raio de 20Km do primeiro utilizador.

In [14]:
q4 = """
SELECT
    m.movieId,
    m.title,
    ROUND(AVG(r.rating), 4) AS averageRating
FROM movies AS m
INNER JOIN ratings AS r ON m.movieId = r.movieId
INNER JOIN users AS u ON r.userId = u.userId
INNER JOIN users AS ref ON ref.userId = 1
WHERE u.userId <> 1
AND ST_DISTANCE_SPHERE(
    POINT(u.longitude, u.latitude),
    POINT(ref.longitude, ref.latitude)
) < 20000
GROUP BY m.movieId, m.title
ORDER BY averageRating DESC, m.movieId ASC;
"""

resultado_q4 = consultar(q4)

print("Número de resultados:", len(resultado_q4))
display(resultado_q4)

Número de resultados: 1902


,movieId,title,averageRating
0,85,Angels and Insects,5.0000
1,232,Eat Drink Man Woman (Yin shi nan nu),5.0000
2,296,Pulp Fiction,5.0000
3,304,Roommates,5.0000
4,353,"Crow, The",5.0000
...,...,...,...
1897,156607,The Huntsman Winter's War,0.5000
1898,156726,Hush,0.5000
1899,161582,Hell or High Water,0.5000
1900,166461,Moana,0.5000


## Q5 — Consulta avançada
Múltiplas junções, filtragem, agregação, agrupamento e ordenação. Consultar os 10 filmes de comédia com melhor classificação média e pelo menos 100 avaliações.

In [15]:
q5 ="""
SELECT m.movieId, m.title,
    ROUND(AVG(r.rating), 4) AS averageRating,
    COUNT(*) AS totalRatings
FROM movies AS m
INNER JOIN ratings AS r ON m.movieId = r.movieId
INNER JOIN movie_genres AS mg ON m.movieId = mg.movieId
INNER JOIN genres AS g ON mg.genreId = g.genreId
WHERE g.genre = 'Comedy'
GROUP BY m.movieId, m.title
HAVING COUNT(*) >= 100
ORDER BY averageRating DESC, m.movieId ASC
LIMIT 10;
"""
resultado_q5 = consultar(q5)

print("Número de resultados:", len(q5))
display(resultado_q5)

Número de resultados: 403


,movieId,title,averageRating,totalRatings
0,1197,"Princess Bride, The",4.2266,139
1,296,Pulp Fiction,4.1914,303
2,4973,"Amelie (Fabuleux destin d'Amélie Poulain, Le)",4.1849,119
3,1136,Monty Python and the Holy Grail,4.1716,134
4,356,Forrest Gump,4.1641,329
5,608,Fargo,4.1160,181
6,778,Trainspotting,4.0392,102
7,1270,Back to the Future,4.0355,169
8,6377,Finding Nemo,3.9640,139
9,1265,Groundhog Day,3.9433,141


# Operações CRUD
Inserção, atualização e eliminação de dados de teste.

In [16]:
testUserId = 999999
testMovieId = 999999

def executar(sql, params=()):
    try:
        with connection.cursor() as cursor:
            cursor.execute(sql, params)
        connection.commit()
    except Exception:
        connection.rollback()
        raise

def verificar_ids(ids):
    for i in ids:
        if not consultar(f"SELECT userId FROM users WHERE userId = {i}").empty:
            raise ValueError(f"Utilizador {i} já existe.")
        if not consultar(f"SELECT movieId FROM movies WHERE movieId = {i} OR imdbId = {i} OR tmdbId = {i}").empty:
            raise ValueError(f"Identificador de filme {i} já existe.")
    connection.rollback()

verificar_ids([testUserId])

## I1 — Inserção de um utilizador

In [18]:
executar("INSERT INTO users (userId, longitude, latitude) VALUES (%s, %s, %s)", (testUserId, 40, -8))
display(consultar(f"SELECT * FROM users WHERE userId = {testUserId}"))

,userId,latitude,longitude
0,999999,-8.0,40.0


## I2 — Inserção de um filme

In [19]:
executar("INSERT INTO movies (movieId, title, year, imdbId, tmdbId) VALUES (%s, %s, %s, %s, %s)", (testMovieId, "Test Movie", 2026, testMovieId, testMovieId))
display(consultar(f"SELECT * FROM movies WHERE movieId = {testMovieId}"))

,movieId,title,year,imdbId,tmdbId
0,999999,Test Movie,2026,999999,999999


## I3 — Inserção de uma avaliação

In [20]:
executar("INSERT INTO ratings (userId, movieId, rating, timestamp) VALUES (%s, %s, %s, NOW())", (testUserId, testMovieId, 3.5))
display(consultar(f"SELECT * FROM ratings WHERE userId = {testUserId} AND movieId = {testMovieId}"))

,userId,movieId,rating,timestamp
0,999999,999999,3.5,2026-11-26 14:37:21


## U1 — Atualização do título

In [21]:
executar("UPDATE movies SET title = %s WHERE movieId = %s", ("Test Movie Updated", testMovieId))
display(consultar(f"SELECT * FROM movies WHERE movieId = {testMovieId}"))

,movieId,title,year,imdbId,tmdbId
0,999999,Test Movie Updated,2026,999999,999999


## U2 — Atualização da avaliação

In [22]:
executar("UPDATE ratings SET rating = %s WHERE userId = %s AND movieId = %s", (4.5, testUserId, testMovieId))
display(consultar(f"SELECT * FROM ratings WHERE userId = {testUserId} AND movieId = {testMovieId}"))

,userId,movieId,rating,timestamp
0,999999,999999,4.5,2026-11-26 14:37:21


## D1 — Eliminação da avaliação

In [23]:
executar("DELETE FROM ratings WHERE userId = %s AND movieId = %s", (testUserId, testMovieId))
display(consultar(f"SELECT * FROM ratings WHERE userId = {testUserId} AND movieId = {testMovieId}"))

,userId,movieId,rating,timestamp


## D2 — Eliminação do filme

In [24]:
executar("DELETE FROM movies WHERE movieId = %s", (testMovieId,))
display(consultar(f"SELECT * FROM movies WHERE movieId = {testMovieId}"))

,movieId,title,year,imdbId,tmdbId


## D3 — Eliminação do utilizador

In [25]:
executar("DELETE FROM users WHERE userId = %s", (testUserId,))
display(consultar(f"SELECT * FROM users WHERE userId = {testUserId}"))

,userId,latitude,longitude


# Transações
## T1 — Transação com sucesso (COMMIT)

Limpeza de dados

In [26]:
def limpar_dados_teste(i):

    connection.rollback()

    try:
        with connection.cursor() as cursor:

            # Eliminar avaliações
            cursor.execute("""
                DELETE FROM ratings
                WHERE userId = %s AND movieId = %s
            """, (i, i))

            # Eliminar filme
            cursor.execute("""
                DELETE FROM movies
                WHERE movieId = %s
            """, (i,))

            # Eliminar utilizador
            cursor.execute("""
                DELETE FROM users
                WHERE userId = %s
            """, (i,))

        connection.commit()

        print(f"Dados de teste {i} eliminados.")

    except mysql.connector.Error:
        connection.rollback()
        raise

### Verifica o estado das tabelas após as operações CRUD

In [27]:
def estado(i):

    users = consultar(
        f"SELECT * FROM users WHERE userId = {i}"
    )

    movies = consultar(
        f"SELECT * FROM movies WHERE movieId = {i}"
    )

    ratings = consultar(
        f"SELECT * FROM ratings WHERE movieId = {i}"
    )

    return {
        "users": len(users),
        "movies": len(movies),
        "ratings": len(ratings)
    }

### Testar transação com commit e rollback

Preparação da transação

In [28]:
i = 999991

limpar_dados_teste(i)
verificar_ids([i])

connection.rollback()
connection.start_transaction()

Dados de teste 999991 eliminados.


Execução da transação

In [30]:
try:
    with connection.cursor() as cursor:

        # Inserir um utilizador
        cursor.execute("""
            INSERT INTO users (userId, longitude, latitude)
            VALUES (%s, %s, %s)
        """, (i, 40, -8))

        # Inserir um filme
        cursor.execute("""
            INSERT INTO movies
            (movieId, title, year, imdbId, tmdbId)
            VALUES (%s, %s, %s, %s, %s)
        """, (i, "Transaction Test Movie", 2026, i, i))

        # Inserir uma avaliação
        cursor.execute("""
            INSERT INTO ratings
            (userId, movieId, rating, timestamp)
            VALUES (%s, %s, %s, NOW())
        """, (i, i, 4.5))

    connection.commit()
    print("COMMIT executado com sucesso.")

except mysql.connector.Error as erro:
    connection.rollback()
    print("Erro:", erro)
    raise

COMMIT executado com sucesso.


Verificação dos resultados

In [31]:
resultado = estado(i)

print(resultado)

assert resultado == {
    "users": 1,
    "movies": 1,
    "ratings": 1
}

connection.rollback()

{'users': 1, 'movies': 1, 'ratings': 1}


In [32]:
print(estado(999992))
connection.rollback()

{'users': 0, 'movies': 0, 'ratings': 0}


## T2 — Transação com falha e rollback automático

Preparação da transação

In [33]:
i = 999992

limpar_dados_teste(i)

verificar_ids([i])

connection.rollback()
connection.start_transaction()

Dados de teste 999992 eliminados.


Transação com falha (ROLLBACK)

In [35]:
try:
    with connection.cursor() as cursor:

        # Inserir um utilizador
        cursor.execute("""
            INSERT INTO users (userId, longitude, latitude)
            VALUES (%s, %s, %s)
        """, (i, 40, -8))

        # Inserir um filme
        cursor.execute("""
            INSERT INTO movies
            (movieId, title, year, imdbId, tmdbId)
            VALUES (%s, %s, %s, %s, %s)
        """, (i, "Rollback Test Movie", 2026, i, i))

        # Inserir uma avaliação válida
        cursor.execute("""
            INSERT INTO ratings
            (userId, movieId, rating, timestamp)
            VALUES (%s, %s, %s, NOW())
        """, (i, i, 3.5))

        # Provocar um erro de chave duplicada
        cursor.execute("""
            INSERT INTO ratings
            (userId, movieId, rating, timestamp)
            VALUES (%s, %s, %s, NOW())
        """, (i, i, 4.5))

except mysql.connector.Error as erro:

    connection.rollback()

    print("ROLLBACK executado.")
    print("Código do erro:", erro.errno)
    print("Mensagem:", erro.msg)

    if erro.errno != 1062:
        raise

else:
    connection.rollback()
    raise AssertionError("Era esperado um erro de chave duplicada.")

ROLLBACK executado.
Código do erro: 1062
Mensagem: Duplicate entry '999992-999992' for key 'ratings.PRIMARY'


verificar o resultado de rollback

In [36]:
resultado = estado(i)

print(resultado)

assert resultado == {
    "users": 0,
    "movies": 0,
    "ratings": 0
}

connection.rollback()

{'users': 0, 'movies': 0, 'ratings': 0}


### Remoção dos dados de T1

In [37]:
limpar_dados_teste(999991)

Dados de teste 999991 eliminados.


## Concorrência em MySQL

Pretende-se avaliar o comportamento do MySQL perante
múltiplos clientes concorrentes, analisando os tempos
de resposta, os conflitos e a consistência dos dados.

### Configuração dos clientes

In [38]:
config = {
    "host": "localhost",
    "port": 3306,
    "user": "root",
    "password": getpass("Password MySQL: "),
    "database": "bda_movielens"
}

NUM_CLIENTES = [1, 5, 10]

### C1 — Leituras concorrentes

Execução simultânea da consulta q5 por diferentes
clientes, avaliando os tempos de resposta e a
consistência dos resultados.

In [39]:
def cliente_leitura(id_cliente, barreira):

    # Criar uma ligação independente
    conn = mysql.connector.connect(**config)

    try:
        with conn.cursor() as cursor:

            # Sincronizar os clientes
            barreira.wait()

            # Medir o tempo de resposta
            inicio = perf_counter()

            cursor.execute(q5)
            dados = cursor.fetchall()

            tempo = (perf_counter() - inicio) * 1000

        return {
            "Cliente": id_cliente,
            "Tempo (ms)": tempo,
            "Dados": dados
        }

    finally:
        conn.close()

### Execução dos testes

In [40]:
resultados_c1 = []
resumo_c1 = []

for n in NUM_CLIENTES:

    barreira = Barrier(n, timeout=30)

    # Executar os clientes em simultâneo
    with ThreadPoolExecutor(max_workers=n) as executor:

        resultados = list(executor.map(
            lambda i: cliente_leitura(i, barreira),
            range(1, n + 1)
        ))

    # Verificar a consistência
    consistente = all(
        r["Dados"] == resultados[0]["Dados"]
        for r in resultados
    )

    tempos = [r["Tempo (ms)"] for r in resultados]

    for r in resultados:
        resultados_c1.append({
            "N.º de clientes": n,
            "Cliente": r["Cliente"],
            "Tempo (ms)": round(r["Tempo (ms)"], 2)
        })

    resumo_c1.append({
        "N.º de clientes": n,
        "Tempo médio (ms)": round(sum(tempos) / n, 2),
        "Tempo mínimo (ms)": round(min(tempos), 2),
        "Tempo máximo (ms)": round(max(tempos), 2),
        "Consistência": consistente
    })

    print(f"Teste com {n} cliente(s) concluído.")

ProgrammingError: 1045 (28000): Access denied for user 'root'@'localhost' (using password: YES)

### Resultados individuais

In [ ]:
df_c1 = pd.DataFrame(resultados_c1)
display(df_c1)

### Resumo dos resultados

In [ ]:
df_resumo_c1 = pd.DataFrame(resumo_c1)
display(df_resumo_c1)

#### Análise dos resultados

Com o aumento do número de clientes concorrentes,
observou-se um aumento do tempo médio de resposta,
de 89,23 ms para 168,10 ms.

Apesar do aumento do tempo de resposta, todos os
clientes obtiveram resultados consistentes.

Não foram observados erros durante a execução
das consultas concorrentes.

### C2 — Escritas concorrentes sem conflito

Execução simultânea de operações de atualização
sobre registos distintos, avaliando os tempos de
resposta e a consistência dos dados.

#### Preparação dos dados de teste

Criação de um filme, dez utilizadores e dez avaliações
temporárias para os testes de escrita concorrente,
sem alterar os dados originais do MovieLens.

In [ ]:
id_filme_c2 = 999980
ids_c2 = list(range(999970, 999980))

conn = mysql.connector.connect(**config)

try:
    with conn.cursor() as cursor:

        # Inserir um filme de teste
        cursor.execute("""
            INSERT INTO movies
            (movieId, title, year, imdbId, tmdbId)
            VALUES (%s, %s, %s, %s, %s)
        """, (
            id_filme_c2,
            "Concurrency Test Movie",
            2026,
            id_filme_c2,
            id_filme_c2
        ))

        # Inserir utilizadores e avaliações de teste
        for i in ids_c2:

            cursor.execute("""
                INSERT INTO users (userId, location)
                VALUES (%s, POINT(%s,%s))
            """, (i, 40, -8))

            cursor.execute("""
                INSERT INTO ratings
                (userId, movieId, rating, timestamp)
                VALUES (%s, %s, %s, NOW())
            """, (i, id_filme_c2, 3.0))

    conn.commit()
    print("Dados de teste criados com sucesso.")

except Exception:
    conn.rollback()
    raise

finally:
    conn.close()

#### Função de escrita concorrente

In [ ]:
def cliente_escrita(id_cliente, barreira):

    conn = mysql.connector.connect(**config)

    try:
        with conn.cursor() as cursor:

            # Sincronizar os clientes
            barreira.wait()
            inicio = perf_counter()

            # Atualizar uma avaliação
            cursor.execute("""
                UPDATE ratings
                SET rating = 4.5
                WHERE userId = %s AND movieId = %s
            """, (id_cliente, id_filme_c2))

            if cursor.rowcount != 1:
                raise RuntimeError("Avaliação não atualizada.")

            conn.commit()

            tempo = (perf_counter() - inicio) * 1000

        return {
            "Cliente": id_cliente,
            "Tempo (ms)": tempo,
            "Sucesso": True,
            "Erro": None
        }

    except Exception as erro:
        conn.rollback()

        return {
            "Cliente": id_cliente,
            "Tempo (ms)": None,
            "Sucesso": False,
            "Erro": str(erro)
        }

    finally:
        conn.close()

#### Reposição dos dados de teste

In [ ]:
def repor_dados_c2():

    conn = mysql.connector.connect(**config)

    try:
        with conn.cursor() as cursor:

            cursor.execute("""
                UPDATE ratings
                SET rating = 3.0
                WHERE movieId = %s
            """, (id_filme_c2,))

        conn.commit()

    except Exception:
        conn.rollback()
        raise

    finally:
        conn.close()

#### Verificação da consistência

In [ ]:
def verificar_c2(n):

    conn = mysql.connector.connect(**config)

    try:
        with conn.cursor() as cursor:

            cursor.execute("""
                SELECT userId, rating
                FROM ratings
                WHERE movieId = %s
                ORDER BY userId
            """, (id_filme_c2,))

            dados = cursor.fetchall()

        esperado = [
            (i, 4.5 if pos < n else 3.0)
            for pos, i in enumerate(ids_c2)
        ]

        return dados == esperado

    finally:
        conn.close()

#### Execução dos testes

In [ ]:
resultados_c2 = []
resumo_c2 = []

for n in NUM_CLIENTES:

    # Repor os valores iniciais
    repor_dados_c2()

    barreira = Barrier(n, timeout=30)

    # Executar os clientes em simultâneo
    with ThreadPoolExecutor(max_workers=n) as executor:

        resultados = list(executor.map(
            lambda i: cliente_escrita(i, barreira),
            ids_c2[:n]
        ))

    # Obter os tempos de resposta
    tempos = [
        r["Tempo (ms)"]
        for r in resultados
        if r["Sucesso"]
    ]

    sucessos = sum(r["Sucesso"] for r in resultados)

    # Guardar os resultados individuais
    for r in resultados:
        resultados_c2.append({
            "N.º de clientes": n,
            **r
        })

    # Guardar o resumo do teste
    resumo_c2.append({
        "N.º de clientes": n,
        "Tempo médio (ms)": round(
            sum(tempos) / len(tempos), 2
        ) if tempos else None,
        "Tempo mínimo (ms)": round(
            min(tempos), 2
        ) if tempos else None,
        "Tempo máximo (ms)": round(
            max(tempos), 2
        ) if tempos else None,
        "Sucessos": sucessos,
        "Erros": n - sucessos,
        "Consistência": verificar_c2(n)
    })

    print(f"Teste com {n} cliente(s) concluído.")

#### Resultados individuais

In [ ]:
df_c2 = pd.DataFrame(resultados_c2)
display(df_c2)

#### Resumo dos resultados

In [ ]:
df_resumo_c2 = pd.DataFrame(resumo_c2)
display(df_resumo_c2)

### C3.1 — Escritas concorrentes sem bloqueio explícito

Simulação de três clientes concorrentes a atualizar
a mesma avaliação, com o objetivo de demonstrar
o problema da atualização perdida (Lost Update).

#### Preparação dos dados de teste

In [ ]:
from decimal import Decimal

id_user_c3 = ids_c2[0]
id_filme_c3 = id_filme_c2

valor_inicial = Decimal("3.0")
incremento = Decimal("0.5")
num_clientes_c3 = 3

valor_esperado = valor_inicial + incremento * num_clientes_c3

In [ ]:
def cliente_sem_bloqueio(id_cliente, barreira):

    conn = mysql.connector.connect(**config)

    try:
        with conn.cursor() as cursor:

            conn.start_transaction()
            inicio = perf_counter()

            # Ler o valor atual
            cursor.execute("""
                SELECT rating
                FROM ratings
                WHERE userId = %s AND movieId = %s
            """, (id_user_c3, id_filme_c3))

            valor = cursor.fetchone()[0]

            # Sincronizar os clientes após a leitura
            barreira.wait()

            novo_valor = valor + incremento

            # Atualizar a avaliação
            cursor.execute("""
                UPDATE ratings
                SET rating = %s
                WHERE userId = %s AND movieId = %s
            """, (novo_valor, id_user_c3, id_filme_c3))

            conn.commit()

            tempo = (perf_counter() - inicio) * 1000

        return {
            "Cliente": id_cliente,
            "Valor lido": valor,
            "Valor escrito": novo_valor,
            "Tempo (ms)": round(tempo, 2),
            "Sucesso": True,
            "Erro": None
        }

    except Exception as erro:
        conn.rollback()

        return {
            "Cliente": id_cliente,
            "Sucesso": False,
            "Erro": str(erro)
        }

    finally:
        conn.close()

### Execução do teste

In [ ]:
repor_dados_c2()

barreira_c3 = Barrier(num_clientes_c3, timeout=30)

with ThreadPoolExecutor(max_workers=num_clientes_c3) as executor:

    resultados_c31 = list(executor.map(
        lambda i: cliente_sem_bloqueio(i, barreira_c3),
        range(1, num_clientes_c3 + 1)
    ))

df_c31 = pd.DataFrame(resultados_c31)
display(df_c31)

### Verificação do resultado

In [ ]:
def obter_rating_c3():

    conn = mysql.connector.connect(**config)

    try:
        with conn.cursor() as cursor:

            cursor.execute("""
                SELECT rating
                FROM ratings
                WHERE userId = %s AND movieId = %s
            """, (id_user_c3, id_filme_c3))

            return cursor.fetchone()[0]

    finally:
        conn.close()

In [ ]:
valor_final_c31 = obter_rating_c3()

consistencia_c31 = valor_final_c31 == valor_esperado

print("Valor inicial:", valor_inicial)
print("Valor esperado:", valor_esperado)
print("Valor final:", valor_final_c31)
print("Consistência:", consistencia_c31)

### C3.2 — Escritas concorrentes com bloqueio de linhas

Utilização do mecanismo SELECT FOR UPDATE para bloquear
o registo durante a transação, evitando o problema
da atualização perdida.

In [ ]:
def cliente_com_bloqueio(id_cliente, barreira):

    conn = mysql.connector.connect(**config)

    try:
        with conn.cursor() as cursor:

            # Sincronizar os clientes antes do bloqueio 
            barreira.wait()

            inicio = perf_counter()
            conn.start_transaction()

            # Ler e bloquear o registo
            cursor.execute("""
                SELECT rating
                FROM ratings
                WHERE userId = %s AND movieId = %s
                FOR UPDATE
            """, (id_user_c3, id_filme_c3))

            valor = cursor.fetchone()[0]

            novo_valor = valor + incremento

            # Atualizar a avaliação
            cursor.execute("""
                UPDATE ratings
                SET rating = %s
                WHERE userId = %s AND movieId = %s
            """, (novo_valor, id_user_c3, id_filme_c3))

            conn.commit()

            tempo = (perf_counter() - inicio) * 1000

        return {
            "Cliente": id_cliente,
            "Valor lido": valor,
            "Valor escrito": novo_valor,
            "Tempo (ms)": round(tempo, 2),
            "Sucesso": True,
            "Erro": None
        }

    except Exception as erro:
        conn.rollback()

        return {
            "Cliente": id_cliente,
            "Sucesso": False,
            "Erro": str(erro)
        }

    finally:
        conn.close()

### Execução de teste 

In [ ]:
repor_dados_c2()

barreira_c3 = Barrier(num_clientes_c3, timeout=30)

with ThreadPoolExecutor(max_workers=num_clientes_c3) as executor:

    resultados_c32 = list(executor.map(
        lambda i: cliente_com_bloqueio(i, barreira_c3),
        range(1, num_clientes_c3 + 1)
    ))

df_c32 = pd.DataFrame(resultados_c32)
display(df_c32)

### Verificação do resultado

In [ ]:
valor_final_c32 = obter_rating_c3()

consistencia_c32 = valor_final_c32 == valor_esperado

print("Valor inicial:", valor_inicial)
print("Valor esperado:", valor_esperado)
print("Valor final:", valor_final_c32)
print("Consistência:", consistencia_c32)

### Análise comparativa dos testes de concorrência

Comparação dos tempos de resposta, do sucesso das
operações e da consistência dos dados nos diferentes
cenários de acesso concorrente.

In [ ]:
resumo_final = pd.concat([
    df_resumo_c1.assign(Teste="C1 — Leituras"),
    df_resumo_c2.assign(Teste="C2 — Escritas sem conflito"),
    pd.DataFrame([
        {
            "Teste": "C3.1 — Sem bloqueio",
            "N.º de clientes": num_clientes_c3,
            "Tempo médio (ms)": df_c31["Tempo (ms)"].mean(),
            "Sucessos": df_c31["Sucesso"].sum(),
            "Erros": (~df_c31["Sucesso"]).sum(),
            "Consistência": consistencia_c31
        },
        {
            "Teste": "C3.2 — Com bloqueio",
            "N.º de clientes": num_clientes_c3,
            "Tempo médio (ms)": df_c32["Tempo (ms)"].mean(),
            "Sucessos": df_c32["Sucesso"].sum(),
            "Erros": (~df_c32["Sucesso"]).sum(),
            "Consistência": consistencia_c32
        }
    ])
], ignore_index=True)

display(resumo_final[[
    "Teste",
    "N.º de clientes",
    "Tempo médio (ms)",
    "Consistência"
]].round(2))

In [ ]:
df_comparacao_c3 = pd.DataFrame({
    "Teste": ["Sem bloqueio", "Com bloqueio"],
    "N.º de clientes": [num_clientes_c3] * 2,
    "Valor inicial": [valor_inicial] * 2,
    "Valor esperado": [valor_esperado] * 2,
    "Valor final": [valor_final_c31, valor_final_c32],
    "Consistência": [consistencia_c31, consistencia_c32]
})

display(df_comparacao_c3)

### Remoção dos dados de teste

Eliminação dos registos temporários utilizados nos
testes de concorrência, preservando os dados
originais do MovieLens.

In [ ]:
conn = mysql.connector.connect(**config)

try:
    with conn.cursor() as cursor:

        # Eliminar as avaliações de teste
        cursor.executemany("""
            DELETE FROM ratings
            WHERE userId = %s AND movieId = %s
        """, [(i, id_filme_c2) for i in ids_c2])

        # Eliminar o filme de teste
        cursor.execute("""
            DELETE FROM movies
            WHERE movieId = %s
        """, (id_filme_c2,))

        # Eliminar os utilizadores de teste
        cursor.executemany("""
            DELETE FROM users
            WHERE userId = %s
        """, [(i,) for i in ids_c2])

    conn.commit()
    print("Dados de teste eliminados com sucesso.")

except Exception:
    conn.rollback()
    raise

finally:
    conn.close()

## Encerramento da ligação

In [ ]:
connection.close()